# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('GEMINI_API_KEY')

if api_key and api_key.startswith('AIza') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gemini-3.1-flash-lite'
gemini = OpenAI(base_url="https://generativelanguage.googleapis.com/v1beta/", api_key=api_key)

API key looks good so far


In [3]:
links = fetch_website_links("https://edwarddonner.com")
links

['#wp--skip-link--target',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https:/

## First step: Have Gemini-3.1-flash-lite figure out which links are relevant

### Use a call to gemini-3.1-flash-lite to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [4]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [5]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [6]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#wp--skip-link--target
https://edwarddonner.com/avatar/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/avatar/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17

In [7]:
def select_relevant_links(url):
    response = gemini.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    print(f"Raw response from Gemini:\n{result}")
    links = json.loads(result)
    return links
    

In [8]:
select_relevant_links("https://edwarddonner.com")

Raw response from Gemini:
{
  "links": [
    {
      "type": "about page",
      "url": "https://edwarddonner.com/about-me-and-about-nebula/"
    },
    {
      "type": "company website",
      "url": "https://nebula.io/?utm_source=ed&utm_medium=referral"
    },
    {
      "type": "curriculum page",
      "url": "https://edwarddonner.com/curriculum/"
    }
  ]
}


{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'company website',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'},
  {'type': 'curriculum page', 'url': 'https://edwarddonner.com/curriculum/'}]}

In [11]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = gemini.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [12]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gemini-3.1-flash-lite
Found 4 relevant links


{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'company page',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'},
  {'type': 'curriculum', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'professional profile',
   'url': 'https://www.linkedin.com/in/eddonner/'}]}

In [13]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-3.1-flash-lite
Found 8 relevant links


{'links': [{'type': 'about page', 'url': 'https://huggingface.co/huggingface'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'brand page', 'url': 'https://huggingface.co/brand'},
  {'type': 'blog page', 'url': 'https://huggingface.co/blog'},
  {'type': 'documentation page', 'url': 'https://huggingface.co/docs'},
  {'type': 'linkedin page',
   'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'github page', 'url': 'https://github.com/huggingface'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to Gemini-3.1-flash-lite

In [14]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [15]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gemini-3.1-flash-lite
Found 6 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Papers
Hardware
Learn
Discord
Forum
GitHub
Solutions
Team & Enterprise
Hugging Face PRO
Enterprise Support
Inference Providers
Inference Endpoints
Storage Buckets
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
Cloudflare/clef
Updated
6 days ago
•
7.26k
•
1.72k
autotrust/JEV-27B-VL
Updated
4 days ago
•
1.53M
•
1.15k
Aleph-Alpha/Kolibri-1
Updated
4 days ago
•
4.14k
•
735
autotrust/GEV-26B-Decide
Updated
4 days ago
•
855k
•
844
abenzerps/Qwen-Image-2.1-Uncensored-GGUF
Updated
9 days 

In [23]:
# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""


In [17]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [18]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-3.1-flash-lite
Found 7 relevant links


"\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nHardware\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nCloudflare/clef\nUpdated\n6 days ago\n•\n7.26k\n•\n1.72k\nautotrust/JEV-27B-VL\nUpdated\n4 days ago\n•\n1.53M\n•\n1.

In [19]:
def create_brochure(company_name, url):
    response = gemini.chat.completions.create(
        model="gemini-3.1-flash-lite",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [20]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-3.1-flash-lite
Found 5 relevant links


# Hugging Face: The AI Community Building the Future

## Who We Are
Hugging Face is the leading global platform for the machine learning community. We serve as the central hub where researchers, developers, and organizations come together to create, discover, and collaborate on the next generation of artificial intelligence. By democratizing access to AI, we are building the future of open-source technology.

## What We Offer
Our platform provides an expansive ecosystem designed to support the entire lifecycle of machine learning development:

*   **Models & Datasets:** Explore a massive library of over 2 million models and 500,000 datasets, enabling developers to build upon the work of the global AI community.
*   **Spaces:** A collaborative environment to build, host, and share interactive AI applications.
*   **Enterprise Solutions:** We provide robust tools for businesses, including inference endpoints, storage buckets, and dedicated enterprise support, allowing teams to integrate state-of-the-art AI into their own infrastructure securely and efficiently.
*   **Learning & Research:** Stay at the forefront of the industry with our blog, daily research papers, and educational tracks designed to advance the state of open-source AI.

## Our Culture: Community-First
Collaboration is at the heart of everything we do. Hugging Face is more than just a tool; it is a vibrant ecosystem where knowledge is shared freely. Whether it is through our active Discord server, community forums, or open-source repositories on GitHub, we prioritize transparency, global cooperation, and the continuous advancement of machine learning technology. We are committed to the philosophy that AI should be accessible, collaborative, and built in the open.

## Why Work With Us?
Hugging Face is at the center of the global AI revolution. Joining our team means:
*   **Impact:** Working on tools that power millions of developers and organizations worldwide.
*   **Innovation:** Being part of a team that publishes original research and pushes the boundaries of AI agents, robotics, and model architecture.
*   **Collaboration:** Working alongside the brightest minds in the machine learning field to shape the future of global technology.

## Get Started Today
Whether you are a researcher looking to publish your findings, a developer building a new AI-powered application, or an enterprise looking to deploy custom models, Hugging Face provides the infrastructure you need to succeed.

**Join the movement.**
Visit us at [huggingface.co](https://huggingface.co) to explore our platform, contribute to open-source, or learn more about our enterprise plans.

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [21]:
def stream_brochure(company_name, url):
    stream = gemini.chat.completions.create(
        model="gemini-3.1-flash-lite",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [22]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-3.1-flash-lite
Found 8 relevant links


# Hugging Face: The Home of Machine Learning

Hugging Face is the central collaboration platform where the global machine learning community builds the future. By providing a unified space to create, discover, and share innovation, Hugging Face has become the essential infrastructure for the modern AI ecosystem.

## Empowering the AI Community
Hugging Face democratizes artificial intelligence by removing barriers to entry. Our platform serves as a massive, open repository where developers and researchers can access and contribute to the evolution of technology.

**Our Core Ecosystem includes:**
* **2 Million+ Models:** A vast library of pre-trained models ready for deployment.
* **500,000+ Datasets:** Comprehensive data resources to fuel your machine learning projects.
* **1 Million+ AI Apps:** Interactive applications built in Spaces that showcase the real-world utility of modern models.
* **HuggingChat:** An intuitive interface for engaging with cutting-edge conversational AI.

## Culture and Values
At Hugging Face, our culture is defined by **radical openness and collaboration**. We believe that the best AI is built when the world works together. Whether you are an individual researcher, an open-source contributor, or an enterprise engineer, our community is a space for transparent knowledge sharing, peer review, and collective progress. We are deeply committed to the growth of the open-source ecosystem, fostering an environment where curiosity and technical excellence lead to the next generation of breakthroughs.

## Solutions for Enterprise
For businesses looking to integrate AI into their operations, Hugging Face provides robust enterprise-grade solutions. We bridge the gap between experimental research and production-ready applications through:
* **Enterprise Support:** Dedicated guidance for scaling AI projects.
* **Inference Endpoints & Providers:** Reliable, high-performance infrastructure to host and serve models securely.
* **Storage Buckets:** Seamless data management for complex machine learning workflows.
* **Professional Tiers:** Tools designed to accelerate team productivity and deployment cycles.

## Careers and Growth
Building the future requires a diverse team of visionaries. Hugging Face is constantly seeking passionate individuals who are dedicated to the ethics of open AI and the advancement of machine learning. We offer a unique opportunity to work at the intersection of industry-leading research and global infrastructure. If you are driven by the desire to build tools that empower millions of developers worldwide, we invite you to join us in shaping the landscape of global technology.

**Join the movement.** Whether you are deploying your first model or scaling an enterprise AI strategy, you are building the future at Hugging Face.

*Visit us at [huggingface.co](https://huggingface.co) to start collaborating today.*

In [24]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-3.1-flash-lite
Found 8 relevant links


# Welcome to Hugging Face: Where AI Gets Its Heart (and a Smilie Face)

Ever wondered where the "brains" of the modern AI revolution hang out? It’s not in a windowless server room guarded by robots—it’s at **Hugging Face**. 

We aren't just building the future; we’re hosting it, sharing it, and probably making a few bad puns about "Transformers" along the way. Think of us as the town square for the Machine Learning world, only with better documentation and significantly less yelling.

### What is this place?
In simple terms, we are the GitHub of Artificial Intelligence. If you’re a developer, researcher, or just someone who thinks "artificial neural networks" sounds like a cool band name, you’ve come to the right place. 

We provide the infrastructure where the magic happens:
* **Models:** Over 2 million of them. Yes, two million. We have models for everything from generating art to analyzing medical records.
* **Datasets:** Because an AI without data is like a chef without groceries. We’ve got over 500,000 datasets waiting to be turned into the next big thing.
* **Spaces:** Want to show off that fancy AI app you just built? Deploy it to a Space and let the world play with it. It’s like a digital science fair, but everyone wins.

### The Culture: Nerdy, Open, and Obsessed
At Hugging Face, our company culture is built on "Open Source" values. We believe that if you want to build the future, you shouldn't do it behind a locked door. We are a community of builders, tinkerers, and researchers who are obsessed with pushing the boundaries of what a computer can do. 

We love agents that think, bots that code, and discussions about the ethics of AI that actually go somewhere. If you can handle a high-speed environment where the "state of the art" changes every Tuesday, you’ll fit right in.

### Why join the "Hugging" Team?
Are you tired of explaining to your relatives that you don't "fix printers"? Do you want to work on tools that millions of people actually use? 

* **Impact:** Your code today might power the next breakthrough in climate research, healthcare, or—let’s be honest—an AI that generates synchronized soundtracks for your cat videos.
* **Community:** You’ll be working alongside the brightest minds in the global AI ecosystem.
* **Perks:** Besides being at the center of the hottest industry on the planet, you get to work with a team that has a sense of humor. We’re literally named after an emoji. How much more professional could we be?

### For the Investors
We’re not just a platform; we’re the backbone of the global open-source AI infrastructure. We’re scaling as fast as the models themselves, providing the enterprise solutions, inference endpoints, and compute power that companies need to stop reinventing the wheel and start deploying intelligence. We’re in it for the long haul, the compute, and the code.

### Come Say Hi!
Whether you’re looking to host your first model, build an agent that does your chores, or join our team of world-class engineers, don't be a stranger. 

**Join the AI community building the future. Join Hugging Face.**

*Warning: We are not responsible for any existential crises caused by your newly created chatbots.*

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>